# 04 · Feature engineering

Goal: give the model what a sharp handicapper looks at. How good is this offense **after
adjusting for who it played**, how good is the opponent's defense (adjusted the same
way), and what's the context (home, rest, travel, weather)?

**Golden rule (as-of):** every feature for a game uses **only games played before it**.
Using the game's own result, or later games, is leakage. The model will look great in the
backtest and fail in real life.

Output: `data/processed/team_games.parquet`, one row per team per game, with the target
`points` and every feature. See `docs/pipeline.md` for the full feature plan.

In [ ]:
import numpy as np
import pandas as pd

from canes_cfb.paths import PROCESSED, RAW

games = pd.read_parquet(RAW / "games.parquet")

## 1. Team-game table
One row per team per game (`team`, `opponent`, `is_home`, `neutral_site`, `points`,
`points_allowed`, points by quarter). This is the shape the points model trains on.

## 2. Raw form (the naive version)
Rolling points scored and allowed (last 3 and last 5 games, season to date). This is the
baseline. It's what the adjusted features must improve on.

## 3. Opponent-adjusted offense and defense (the core idea)
Scoring 40 against a bad defense is not the same as scoring 40 against a good one.
Approach: before each week, fit a ridge regression on all prior games:

`points = league_avg + offense[team] − defense[opponent] + home_edge`

This gives every team an adjusted offense and defense rating as of that week. The
features for a game are the team's offense rating vs. the opponent's defense rating.

## 4. Recency and early-season priors
Weight recent games more (exponential decay). In weeks 1–4 there's little current-season
data, so ratings start from last season's final rating, shrunk toward the average.

## 5. Context
Home/away/neutral, rest days, travel distance, conference game, week of season, dome.

## 6. Period shares (for halves and quarters)
Each team's historical share of points by quarter and half (fast starters vs. strong
finishers), as-of.

## 7. Leakage checks and save
Recompute features for a sample of games using only data before kickoff and assert
they're identical. Then save.